# [1] Google Colab Backend for Multimodal AI (POLISHED VERSION)
### (Merged to fix empty Illustration Board and Whisper errors)

---

In [ ]:
# [0] ОЧИСТКА ПОРТА & GPU
!fuser -k 9073/tcp
import torch
print("--- ПОРТ 9073 ОЧИЩЕН. ---", flush=True)
print(f"--- GPU: {torch.cuda.get_device_name(0)} ---" if torch.cuda.is_available() else "--- ВНИМАНИЕ: GPU НЕ НАЙДЕН ---", flush=True)

In [ ]:
# [1] Установка библиотек (займет ~5 мин)
!pip install -q transformers torch torchvision torchaudio aiohttp aiohttp_cors Pillow numpy kokoro spacy qwen-vl-utils accelerate bitsandbytes
!python -m spacy download en_core_web_sm -q
!npm install -g localtunnel -q
print("--- Установка завершена ---", flush=True)

In [ ]:
# [2] ПАРОЛЬ ДЛЯ ТОННЕЛЯ (IPV4)
import requests
ip = requests.get('https://loca.lt/mytunnelpassword').text.strip()
print(f"--- ВАШ ПАРОЛЬ ДЛЯ ТОННЕЛЯ: {ip} ---", flush=True)

In [ ]:
# [3] ФИНАЛЬНЫЙ ЗАПУСК СЕРВЕРА (Всё в одной ячейке)
import os, subprocess, threading, json, base64, torch, numpy as np, logging, sys, io, time, asyncio
from PIL import Image
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from kokoro import KPipeline
from aiohttp import web

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', stream=sys.stdout)
logger = logging.getLogger(__name__)

class AudioSegmentDetector:
    def __init__(self, sample_rate=16000, energy_threshold=0.008, silence_duration=0.8, min_speech_duration=0.8, max_speech_duration=15):
        self.sample_rate = sample_rate; self.energy_threshold = energy_threshold; self.silence_samples = int(silence_duration * sample_rate)
        self.min_speech_samples = int(min_speech_duration * sample_rate); self.max_speech_samples = int(max_speech_duration * sample_rate)
        self.audio_buffer = bytearray(); self.is_speech_active = False; self.silence_counter = 0; self.speech_start_idx = 0
        self.lock = asyncio.Lock(); self.segment_queue = asyncio.Queue(); self.tts_playing = False; self.tts_lock = asyncio.Lock()
        self.last_print_time = time.time()
    async def set_tts_playing(self, is_playing): 
        async with self.tts_lock: self.tts_playing = is_playing
    async def add_audio(self, audio_bytes):
        async with self.lock:
            async with self.tts_lock: 
                if self.tts_playing: return None
            self.audio_buffer.extend(audio_bytes)
            audio_array = np.frombuffer(audio_bytes, dtype=np.int16).astype(np.float32) / 32768.0
            if len(audio_array) > 0:
                energy = np.sqrt(np.mean(audio_array**2))
                if time.time() - self.last_print_time > 2:
                    print(f"[Радар шума] {energy:.6f} / {self.energy_threshold}", flush=True)
                    self.last_print_time = time.time()
                if not self.is_speech_active and energy > self.energy_threshold:
                    self.is_speech_active = True; self.speech_start_idx = max(0, len(self.audio_buffer) - len(audio_bytes))
                    print(f"--- РЕЧЬ НАЧАЛАСЬ ({energy:.4f}) ---", flush=True)
                elif self.is_speech_active:
                    if energy > self.energy_threshold: self.silence_counter = 0
                    else:
                        self.silence_counter += len(audio_array)
                        if self.silence_counter >= self.silence_samples:
                            end_idx = len(self.audio_buffer) - self.silence_counter
                            seg = bytes(self.audio_buffer[self.speech_start_idx:end_idx])
                            self.is_speech_active = False; self.silence_counter = 0; self.audio_buffer = self.audio_buffer[end_idx:]
                            if len(seg) >= self.min_speech_samples * 2: 
                                print("--- РЕЧЬ ЗАКОНЧИЛАСЬ. Обработка... ---", flush=True)
                                await self.segment_queue.put(seg)
            return None
    async def get_next_segment(self): 
        try: return await asyncio.wait_for(self.segment_queue.get(), timeout=0.1)
        except asyncio.TimeoutError: return None

class WhisperTranscriber:
    def __init__(self):
        self.device = "cuda:0" if torch.cuda.is_available() else "cpu"
        self.torch_dtype = torch.float16 if self.device != "cpu" else torch.float32
        self.model = AutoModelForSpeechSeq2Seq.from_pretrained("openai/whisper-small", torch_dtype=self.torch_dtype, low_cpu_mem_usage=True, use_safetensors=True).to(self.device)
        self.processor = AutoProcessor.from_pretrained("openai/whisper-small")
    async def transcribe(self, audio_bytes):
        try:
            arr = np.frombuffer(audio_bytes, dtype=np.int16).astype(np.float32) / 32768.0
            if len(arr) < 160: return ""
            input_features = self.processor(arr, sampling_rate=16000, return_tensors="pt").input_features.to(self.device).to(self.torch_dtype)
            with torch.no_grad():
                ids = self.model.generate(input_features, task="transcribe", language="english")
            txt = self.processor.batch_decode(ids, skip_special_tokens=True)[0]
            return txt.strip()
        except Exception as e: return f"Whisper Error: {e}"

class QwenMultimodalProcessor:
    def __init__(self):
        self.device = "cuda:0" if torch.cuda.is_available() else "cpu"; model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
        if torch.cuda.is_available():
            q_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
            self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(model_id, quantization_config=q_config, device_map="auto")
        else: self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.bfloat16, device_map="cpu")
        self.processor = AutoProcessor.from_pretrained(model_id); self.last_image = None; self.last_image_bytes = None; self.lock = asyncio.Lock()
    async def set_image(self, image_data):
        async with self.lock:
            try:
                img = Image.open(io.BytesIO(image_data))
                new_size = (int(img.size[0] * 0.7), int(img.size[1] * 0.7))
                self.last_image = img.resize(new_size, Image.Resampling.LANCZOS); self.last_image_bytes = image_data; return True
            except: return False
    async def generate(self, text):
        async with self.lock:
            try:
                if not self.last_image: return f"Looking for visuals: {text}"
                msg = [{"role": "system", "content": "You are a helpful AI assistant. Be concise and conversational (2 sentences max)."}, {"role": "user", "content": [{"type": "image", "image": self.last_image}, {"type": "text", "text": text}]}]
                ti = self.processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
                inp = self.processor(text=[ti], images=[self.last_image], padding=True, return_tensors="pt").to(self.device)
                gen = self.model.generate(**inp, max_new_tokens=80, do_sample=True, temperature=0.7)
                out = self.processor.batch_decode([g[len(i):] for i,g in zip(inp.input_ids, gen)], skip_special_tokens=True)[0]
                return out
            except Exception as e: return f"Qwen Error: {e}"

class KokoroTTSProcessor:
    def __init__(self):
        self.pipeline = KPipeline(lang_code='a'); self.voice = 'af_sarah'
    async def synthesize_speech(self, text):
        try:
            gen = await asyncio.get_event_loop().run_in_executor(None, lambda: self.pipeline(text, voice=self.voice, speed=1, split_pattern=r'[.!?。！？]+'))
            segs = [a for _,_,a in gen]; return np.concatenate(segs) if segs else None
        except: return None

async def handle_client(websocket):
    print("--- КЛИЕНТ ПОДКЛЮЧЕН! ---", flush=True)
    dtct = AudioSegmentDetector(); tx = WhisperTranscriber(); qw = QwenMultimodalProcessor(); tt = KokoroTTSProcessor()
    
    async def listen():
        while True:
            seg = await dtct.get_next_segment()
            if seg:
                print(f"--- Whisper транскрипция ({len(seg)} байт)... ---", flush=True)
                txt = await tx.transcribe(seg)
                if txt:
                    print(f"Whisper: {txt}", flush=True)
                    await websocket.send_json({"transcription": {"text": txt, "sender": "User", "finished": True}})
                    res = await qw.generate(txt)
                    print(f"Qwen: {res}", flush=True)
                    await websocket.send_json({"text": res})
                    # ПОДКЛЮЧЕНИЕ ILLUSTRATION BOARD
                    if qw.last_image_bytes:
                        await websocket.send_json({
                            "edited_image": {
                                "image": base64.b64encode(qw.last_image_bytes).decode('utf-8'),
                                "mime_type": "image/jpeg",
                                "explanation": res
                            }
                        })
                    await dtct.set_tts_playing(True)
                    try:
                        print("--- Kokoro синтез... ---", flush=True)
                        aud = await tt.synthesize_speech(res)
                        if aud is not None:
                            b64 = base64.b64encode((aud * 32767).astype(np.int16).tobytes()).decode('utf-8')
                            await websocket.send_json({"audio": b64})
                            await asyncio.sleep(len(aud)/24000)
                    finally: await dtct.set_tts_playing(False)
            await asyncio.sleep(0.01)
            
    async def receive():
        print("--- Сервер активен. Слушаем... ---", flush=True)
        async for msg in websocket:
            if msg.type == web.WSMsgType.TEXT:
                try:
                    data = msg.json()
                    if "realtime_input" in data:
                        for c in data["realtime_input"]["media_chunks"]:
                            if c["mime_type"] == "audio/pcm": await dtct.add_audio(base64.b64decode(c["data"]))
                            elif c["mime_type"] == "image/jpeg" and not dtct.tts_playing: 
                                await qw.set_image(base64.b64decode(c["data"]))
                except: pass
            elif msg.type == web.WSMsgType.ERROR: break
            
    await asyncio.gather(receive(), listen(), return_exceptions=True)
    print("--- КЛИЕНТ ОТКЛЮЧЕН ---", flush=True)

async def websocket_handler(request):
    ws = web.WebSocketResponse(); await ws.prepare(request)
    await handle_client(ws); return ws

def start_tunnel_thread():
    p = subprocess.Popen(['lt', '--port', '9073'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
    for line in p.stdout:
        if 'https://' in line: print(f"\nВАША ССЫЛКА (Копируйте её): {line.strip()}\n", flush=True)

threading.Thread(target=start_tunnel_thread, daemon=True).start()

async def start_server():
    app = web.Application(); app.add_routes([web.get('/', websocket_handler)])
    runner = web.AppRunner(app); await runner.setup()
    site = web.TCPSite(runner, '0.0.0.0', 9073)
    print("\n--- Инициализация ИИ (Whisper Direct, Qwen VL, Kokoro)... ---", flush=True)
    await site.start()
    print("--- ВСЁ ГОТОВО! МОЖНО ГОВОРИТЬ. ---", flush=True)
    await asyncio.Future()

try:
    await start_server()
except Exception as e: print(f"Ошибка: {e}", flush=True)